In [20]:
import re
import json
from pathlib import Path
from copy import deepcopy

In [21]:
INPUT_DIR = 'dataset/figma-data/split'
OUTPUT_DIR = 'dataset/figma-data/cleaned'

PRUNE_INVISIBLE = True
DROP_EMPTY_CHILDREN = True

In [22]:
KEEP_BY_TYPE: dict[str, dict[str, set]] = {
    'FRAME': {
        'always': {'id', 'name', 'type'},
        'conditional': {
            'visible',
            'children',
            'layoutMode',
            'itemSpacing',
            'paddingLeft', 'paddingRight', 'paddingTop', 'paddingBottom',
            'primaryAxisAlignItems', 'counterAxisAlignItems',
            'layoutSizingHorizontal', 'layoutSizingVertical',
            'layoutWrap',
            'cornerRadius',
            'rectangleCornerRadii',
            'fills',
        },
    },
    'INSTANCE': {
        'always': {'id', 'name', 'type', 'componentId'},
        'conditional': {
            'visible',
            'children',
            'componentProperties',
            'componentPropertyReferences',
            'isExposedInstance',
        },
    },
    'TEXT': {
        'always': {'id', 'name', 'type'},
        'conditional': {
            'visible',
            'characters',
            'componentPropertyReferences',
            'style',
        },
    },
    'RECTANGLE': {
        'always': {'id', 'name', 'type'},
        'conditional': {
            'visible',
            'cornerRadius',
            'fills',
        },
    },
    'VECTOR': {
        'always': {'id', 'name', 'type'},
        'conditional': {'visible'},
    },
}

DEFAULT_KEEP = {
    'always': {'id', 'name', 'type'},
    'conditional': {'visible', 'children'},
}

TEXT_STYLE_KEEP = {'fontFamily', 'fontWeight', 'fontSize'}

OMIT_WHEN_DEFAULT = {
    'layoutMode': 'NONE',
    'layoutWrap': 'NO_WRAP',
    'itemSpacing': 0,
    'cornerRadius': 0,
    'paddingLeft': 0,
    'paddingRight': 0,
    'paddingTop': 0,
    'paddingBottom': 0,
}

print('Prune invisible nodes:', PRUNE_INVISIBLE)
print('Drop empty children:', DROP_EMPTY_CHILDREN)

print('Configured Node-Typs:', list(KEEP_BY_TYPE.keys()))
print('Configured Edge-Typs:', list(DEFAULT_KEEP.keys()))
print('Configured Text Styles:', list(TEXT_STYLE_KEEP))

print('Omit when default:', OMIT_WHEN_DEFAULT)

Prune invisible nodes: True
Drop empty children: True
Configured Node-Typs: ['FRAME', 'INSTANCE', 'TEXT', 'RECTANGLE', 'VECTOR']
Configured Edge-Typs: ['always', 'conditional']
Configured Text Styles: ['fontFamily', 'fontSize', 'fontWeight']
Omit when default: {'layoutMode': 'NONE', 'layoutWrap': 'NO_WRAP', 'itemSpacing': 0, 'cornerRadius': 0, 'paddingLeft': 0, 'paddingRight': 0, 'paddingTop': 0, 'paddingBottom': 0}


In [23]:
def clean_fills(fills) -> list[dict]:
    if not isinstance(fills, list):
        return []

    result = []

    for fill in fills:
        if not isinstance(fill, dict):
            continue

        if fill.get('type') == 'SOLID' and 'color' in fill:
            color = fill['color']

            result.append({
                'type': 'SOLID',
                'color': {k: round(v, 4) for k, v in color.items()},
            })

    return result

def clean_text_style(style) -> dict:
    if not isinstance(style, dict):
        return {}

    return {k: v for k, v in style.items() if k in TEXT_STYLE_KEEP}

def should_omit(key: str, value) -> bool:
    if value is None:
        return True

    if key in OMIT_WHEN_DEFAULT and value == OMIT_WHEN_DEFAULT[key]:
        return True

    return False

In [24]:
def clean_node(node, prune_invisible: bool = True) -> dict | None:
    if not isinstance(node, dict):
        return None

    # Remove invisible nodes (if configured)
    if prune_invisible and node.get('visible') is False:
        return None

    nt = node.get('type', '_UNKNOWN')
    spec = KEEP_BY_TYPE.get(nt, DEFAULT_KEEP)
    cleaned: dict = {}

    # 1. Always-Keys in strict order
    for key in ('type', 'id', 'name', 'componentId'):
        if key in spec['always'] and key in node:
            cleaned[key] = node[key]

    # 2. Conditional-Keys – without children (come at end)
    for key in sorted(spec['conditional']):
        if key == 'children' or key not in node:
            continue

        value = node[key]

        if should_omit(key, value):
            continue

        # Special cleaning
        if key == 'fills':
            value = clean_fills(value)

            if not value:
                continue
        elif key == 'style':
            value = clean_text_style(value)

            if not value:
                continue

        cleaned[key] = value

    # 3. Children recursively, always at the end
    if 'children' in spec['conditional'] and 'children' in node:
        cleaned_children = []

        for child in node['children'] or []:

            cc = clean_node(child, prune_invisible)

            if cc is not None:
                cleaned_children.append(cc)

        if cleaned_children:
            cleaned['children'] = cleaned_children
        elif not DROP_EMPTY_CHILDREN:
            cleaned['children'] = []

    return cleaned

In [25]:
test_node = {
    'id': '1:1', 'name': 'test', 'type': 'FRAME',
    'scrollBehavior': 'SCROLLS', 'blendMode': 'PASS_THROUGH',
    'layoutMode': 'VERTICAL', 'itemSpacing': 12,
    'paddingLeft': 0, 'paddingRight': 16,
    'effects': [], 'interactions': [],
    'children': [
        {'id': '1:2', 'name': 'hidden', 'type': 'TEXT', 'visible': False, 'characters': 'gone'},
        {'id': '1:3', 'name': 'visible', 'type': 'TEXT', 'characters': 'kept'},
    ],
}

cleaned_test_node = clean_node(test_node)

print('Pre Cleaning :', json.dumps(test_node, indent=2))
print('Post Cleaning:', json.dumps(clean_node(test_node), indent=2))
print(f'Reduction percentage: {100 * (1 - len(json.dumps(cleaned_test_node)) / len(json.dumps(test_node))):.2f}%')

Pre Cleaning : {
  "id": "1:1",
  "name": "test",
  "type": "FRAME",
  "scrollBehavior": "SCROLLS",
  "blendMode": "PASS_THROUGH",
  "layoutMode": "VERTICAL",
  "itemSpacing": 12,
  "paddingLeft": 0,
  "paddingRight": 16,
  "effects": [],
  "interactions": [],
  "children": [
    {
      "id": "1:2",
      "name": "hidden",
      "type": "TEXT",
      "visible": false,
      "characters": "gone"
    },
    {
      "id": "1:3",
      "name": "visible",
      "type": "TEXT",
      "characters": "kept"
    }
  ]
}
Post Cleaning: {
  "type": "FRAME",
  "id": "1:1",
  "name": "test",
  "itemSpacing": 12,
  "layoutMode": "VERTICAL",
  "paddingRight": 16,
  "children": [
    {
      "type": "TEXT",
      "id": "1:3",
      "name": "visible",
      "characters": "kept"
    }
  ]
}
Reduction percentage: 50.38%


In [26]:
def count_keys(obj) -> int:
    if isinstance(obj, dict):
        return len(obj) + sum(count_keys(v) for v in obj.values())
    elif isinstance(obj, list):
        return sum(count_keys(item) for item in obj)

    return 0

INPUT_DIR_PATH = Path(INPUT_DIR)
OUTPUT_DIR_PATH = Path(OUTPUT_DIR)

input_files = sorted(INPUT_DIR_PATH.rglob('*.json'))
print(f'Found {len(input_files)} input files in {INPUT_DIR}')

results = []

for input_file in input_files:
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    cleaned_data = clean_node(data, prune_invisible=PRUNE_INVISIBLE)

    output_file = OUTPUT_DIR_PATH / input_file.relative_to(INPUT_DIR_PATH)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(cleaned_data, f, indent=4, ensure_ascii=False)

    size_before = len(json.dumps(data))
    size_after = len(json.dumps(cleaned_data))

    attributes_before = count_keys(data)
    attributes_after = count_keys(cleaned_data)

    relative_name = str(input_file.relative_to(INPUT_DIR_PATH))
    results.append((relative_name, size_before, size_after, attributes_before, attributes_after, cleaned_data.get('name', '-')))

print('Cleaning Results:')
print(f'\t{"File":<20} {"Before":>10} {"After":>10} {"Reduction":>10} {"Name":<30}')

total_reduction = 0
attributes_before = 0
attributes_after = 0

for filename, size_before, size_after, attrs_before, attrs_after, name in results:
    reduction = 100 * (1 - size_after / size_before) if size_before > 0 else 0
    total_reduction += reduction
    attributes_before += attrs_before
    attributes_after += attrs_after

    print(f'\t{filename:<20} {size_before:>10} {size_after:>10} {reduction:>9.2f}% {attrs_before:>6} -> {attrs_after:<6} {name:<30}')

print(f'Average Reduction: {total_reduction / len(results)}')
print(f'Average Attributes Before: {attributes_before / len(results)}')
print(f'Average Attributes After: {attributes_after / len(results)}')

cleaning_report = {
    'summary': {
        'total_files': len(results),
        'average_reduction_percent': round(total_reduction / len(results), 2),
        'average_attributes_before': round(attributes_before / len(results), 2),
        'average_attributes_after': round(attributes_after / len(results), 2),
    },
    'files': [
        {
            'filename': filename,
            'size_before': size_before,
            'size_after': size_after,
            'reduction_percent': round(100 * (1 - size_after / size_before), 2) if size_before > 0 else 0,
            'attributes_before': attrs_before,
            'attributes_after': attrs_after,
            'name': name,
        }
        for filename, size_before, size_after, attrs_before, attrs_after, name in results
    ],
}

report_path = Path(OUTPUT_DIR) / 'cleaning_report.json'

with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(cleaning_report, f, indent=4, ensure_ascii=False)

print(f'\nReport saved to: {report_path}')


Found 30 input files in dataset/figma-data/split
Cleaning Results:
	File                     Before      After  Reduction Name                          
	hard\1.json              110741      17005     84.64%   4798 -> 796    1 [Bearbeitungs-Dialog]       
	hard\10.json             233758      40720     82.58%  10476 -> 1907   10 [Admin-Panel-Liste]        
	hard\2.json              144889      21432     85.21%   6477 -> 1022   2 [Nutzer-Tabelle]            
	hard\3.json              402648      62123     84.57%  17495 -> 2864   3 [Filter-Leiste]             
	hard\4.json              161051      30090     81.32%   7270 -> 1407   4 [Aktions-Tabelle]           
	hard\5.json              164566      12945     92.13%   7211 -> 623    5 [Step-Formular-Dialog]      
	hard\6.json               56425      10531     81.34%   2491 -> 474    6 [Select] - !Ändern!         
	hard\7.json              164612      24838     84.91%   7148 -> 1200   7 [Benachrichtigungs-Liste]   
	hard\8.json           

In [29]:
from collections import defaultdict

component_properties: dict[str, set[str]] = defaultdict(set)
component_count: dict[str, int] = defaultdict(int)

def walk_for_inventory(node):
    if not isinstance(node, dict):
        return
    if node.get('type') == 'INSTANCE':
        comp_name = node.get('name', '?')
        component_count[comp_name] += 1

        for prop_key in (node.get('componentProperties') or {}).keys():
            component_properties[comp_name].add(prop_key)

    for child in node.get('children', []) or []:
        walk_for_inventory(child)

for relative_name, *_ in results:
    output_path = OUTPUT_DIR_PATH / relative_name
    with open(output_path, 'r', encoding='utf-8') as f:
        walk_for_inventory(json.load(f))

inventory = []
for name in sorted(component_properties.keys()):
    inventory.append({
        'component_name': name,
        'instances_total': component_count[name],
        'property_count': len(component_properties[name]),
        'properties': ', '.join(sorted(component_properties[name])),
    })

print('\nComponent Inventory:')
print(f'\t{"Component Name":<30} {"Instances":>10} {"Properties":>10} {"Property Keys":<40}')

for item in inventory:
    print(f'\t{item["component_name"]:<30} {item["instances_total"]:>10} {item["property_count"]:>10} {item["properties"]:<40}')

inventory_path = Path(OUTPUT_DIR) / 'component_inventory.json'

with open(inventory_path, 'w', encoding='utf-8') as f:
    json.dump(inventory, f, indent=4, ensure_ascii=False)

print(f'Component inventory saved to: {inventory_path}')


Component Inventory:
	Component Name                  Instances Properties Property Keys                           
	_accordion-header                       2          4 Disabled, Focus, Header#4272:33, State  
	_accordion-panel                        2          2 Content Type, Toggle Status             
	_breadcrumb-item                        6          5 Focus, Hover, Icon#7380:18, Show Item Icon#7380:9, Type
	_datatable-body-cell                    8          2 Grid Lines, Size                        
	_datatable-content                      8          2 Content#4293:987, Type                  
	_datatable-header-cell                  7          7 Grid Lines, Header#4295:25, Hover, Selected, Size, Sort Order, Sortable
	_datepicker-cell                       35          5 Day#4432:0, Month#4432:20, State, Type, Year#7014:0
	_datepicker-select                      2          3 Month / Year, State, Text#3997:0        
	_inputnumber-button                     8          2 Icon#6209:0,